In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def img_area(bin_img):
    area=np.count_nonzero(bin_img)
    return area

In [3]:
def img_perimeter(border_img):
    perimeter=np.count_nonzero(border_img)
    return perimeter

In [4]:
def max_d(bin_img):
    minx,miny=10000000,1000000
    maxx,maxy=0,0
    h,w=bin_img.shape
    for x in range(h):
        for y in range(w):
            if(bin_img[x,y]<=0):
                continue
            minx=min(minx,x)
            miny=min(miny,y)
            maxx=max(maxx,x)
            maxy=max(maxy,y)
    return max(maxx-minx,maxy-miny)
            

In [5]:
def find_ab(bin_img):
    contours, _ = cv2.findContours(bin_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnt = max(contours, key=cv2.contourArea)
    if len(cnt) >= 5:
        (x, y), (MA, ma), angel = cv2.fitEllipse(cnt)
        a = max(MA, ma)
        b = min(MA, ma)
    else:
        print("404")
    
    return a, b

In [6]:
def clac_descriptors(bin_img,i):
    k=np.ones((3,3), dtype=np.uint8)
    eroded=cv2.erode(bin_img,k,iterations=1)
    bor_img=bin_img-eroded

    area=img_area(bin_img)
    peri=img_perimeter(bor_img)
    maxd=max_d(bin_img)
    a,b=find_ab(bin_img)

    comp=(peri**2)/area
    form_fact=(4*np.pi*area)/(peri**2)
    # roundness=(4*area)/(np.pi*maxd**2)
    eccentricity=np.sqrt(1-(b/a)**2)

    return comp,form_fact,eccentricity

In [7]:
def euc_dist(t0,t1):
    abs_c=(t0[0]-t1[0])**2
    abs_ff=(t0[1]-t1[1])**2
    abs_r=(t0[2]-t1[2])**2

    return np.sqrt(abs_c+abs_ff+abs_r)

In [ ]:
# def find_cos_sim(t1,t2):

#     dot_product=0
#     for i in range(len(t1)):
#         dot_product+=t1[i]*t2[i]
    
#     mag1=0
#     for i in range(len(t1)):
#         mag1+=t1[i]*t1[i]
#     mag1=np.sqrt(mag1)
    
#     mag2=0
#     for i in range(len(t2)):
#         mag2+=t2[i]*t2[i]
#     mag2 = np.sqrt(mag2)
    
#     if mag1==0 or mag2==0:
#         return 0
    
#     similarity=dot_product/(mag1*mag2)
#     return similarity
def find_cos_sim(t1, t2):
    t1 = np.array(t1)
    t2 = np.array(t2)
    
    dot_product = np.dot(t1, t2)
    mag1 = np.linalg.norm(t1)
    mag2 = np.linalg.norm(t2)
    
    if mag1 == 0 or mag2 == 0:
        return 0
    
    similarity = dot_product / (mag1 * mag2)
    return similarity

In [9]:
def kl_div(t1,t2):
    p=np.array(t1)
    q=np.array(t2)
    p=p/np.sum(p)
    q=q/np.sum(q)

    return np.sum(p*np.log(p/q))

In [10]:
def sim_matrix(train_images, test_images):
    train_descriptors=[]
    test_descriptors=[]

    for i,img in enumerate(train_images):
        co,ff,rd=clac_descriptors(img,i)
        train_descriptors.append((co,ff,rd))
    
    for i, img in enumerate(test_images):
        co,ff,rd=clac_descriptors(img,i)
        test_descriptors.append((co,ff,rd))
    
    sim_mat=[]
    for i,test_d in enumerate(test_descriptors):
        sim_row=[]
        for j,train_d in enumerate(train_descriptors):
            dist=kl_div(test_d,train_d)
            sim_row.append(dist)
        sim_mat.append(sim_row)
    
    print("train",train_descriptors)
    print("test",test_descriptors)
    print("sim",sim_mat)
    
    print("Similarity Matrix\n")
    print("\t",end="")
    for j in range(len(train_images)):
        print(f"Train{j + 1}",end="\t")
    print()

    for i in range(len(test_images)):
        print(f"Test {i + 1}\t",end="")
        for j in range(len(train_images)):
            similarity_val=sim_mat[i][j]   
            print(f"{similarity_val:.5f}",end="\t")
        print()
    
    return sim_mat   

In [11]:
train_images=[
    cv2.imread('../test_assets/c1.jpg',0),
    cv2.imread('../test_assets/t1.jpg',0),
    cv2.imread('../test_assets/p1.png',0)
]
test_images=[
    cv2.imread('../test_assets/c2.jpg',0),
    cv2.imread('../test_assets/t2.jpg',0),
    cv2.imread('../test_assets/p2.png',0),
    cv2.imread('../test_assets/st.jpg',0),
]

train_kl=[
    cv2.imread("../test_assets/c1.jpg",0),
    cv2.imread("../test_assets/st.jpg",0),
    cv2.imread("../test_assets/p1.png",0),

]
test_kl=[
    cv2.imread("../test_assets/c1.jpg",0),
    cv2.imread("../test_assets/st.jpg",0),
    cv2.imread("../test_assets/p1.png",0),
    cv2.imread("../test_assets/t1.jpg",0),

]
result=sim_matrix(train_kl,test_kl)

train [(412.4809497515185, 0.030465335725029834, np.float64(0.23538678410688063)), (1435.6794608998669, 0.00875290826162737, np.float64(0.26245805665265565)), (21.2998585572843, 0.5899743691049827, np.float64(0.43403688561658815))]
test [(412.4809497515185, 0.030465335725029834, np.float64(0.23538678410688063)), (1435.6794608998669, 0.00875290826162737, np.float64(0.26245805665265565)), (21.2998585572843, 0.5899743691049827, np.float64(0.43403688561658815)), (404.8762278978389, 0.03103756098402004, np.float64(0.8952368076488147))]
sim [[np.float64(0.0), np.float64(0.00037788202010675444), np.float64(0.043835258553432746)], [np.float64(0.00023215523002487507), np.float64(0.0), np.float64(0.04585428671999349)], [np.float64(0.17984153009227088), np.float64(0.2674406755224278), np.float64(0.0)], [np.float64(0.0013500509397788567), np.float64(0.0035966972834631456), np.float64(0.039320856575849016)]]
Similarity Matrix

	Train1	Train2	Train3	
Test 1	0.00000	0.00038	0.04384	
Test 2	0.00023	0.